# 1. Title and context — Shared-core smoke test (archival)

This notebook is **step 1** in `config/notebook_plan.json`. It answers: *Does the checked-out public governance engine evaluate a fixed synthetic case end-to-end in **replay_mode** and emit the two hash-locked artefacts?*

**Pipeline position:** archival shared core → consumed by `python reproduce_all.py` → manifest → `config/expected_outputs.json` validation.


# 2. Why this matters (domain, non-technical)

Clinical and high-stakes AI systems are increasingly judged against **governance gates** (safety, evidence, fairness signals, calibration, traceability) before deployment or update. This notebook does not score a real product; it proves that the **reference implementation** in `engine/corrected_public_engine_v1_1.py` runs on a transparent, frozen scenario and that the resulting **CSV and summary text** can be audited byte-for-byte against a public manifest.

**Outputs:** one table row plus a short narrative file—both named in `config/trace_map.json`.


# 3. Deterministic setup — environment assumptions

- **Dependencies:** `requirements.txt` / `environment.yml`, including **`ipywidgets>=8.0.0`** for bounded exploration UI.
- **Hash seed:** `reproduce_all.py` sets `PYTHONHASHSEED` from `config/harness_settings.json` before notebooks run.
- **Restart-and-run-all:** safe if executed top-to-bottom; exploration widgets only **display** alternate predefined cases. **Canonical files on disk** are written only from the fixed `ARCHIVAL_CASE` in section 7.


In [1]:
from __future__ import annotations

import csv
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import clear_output, display


def repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in (start, *start.parents):
        if (p / "engine" / "corrected_public_engine_v1_1.py").is_file():
            return p
    raise RuntimeError(
        "Repository root not found (expected engine/corrected_public_engine_v1_1.py)."
    )


ROOT = repo_root()
sys.path.insert(0, str(ROOT / "engine"))
import corrected_public_engine_v1_1 as eng

OUT_TAB = ROOT / "outputs" / "tables"
OUT_FIG = ROOT / "outputs" / "figures"
OUT_TAB.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)


# 4. Data / inputs — explicit declarations

- **`ARCHIVAL_CASE`:** the **only** case dict used for **contract outputs** (must match the hash-locked release).
- **`EXPLORATION_CASES`:** additional **predefined** scenarios for the dropdown; bounded set, no free-form input.


In [2]:
ARCHIVAL_CASE = {
    "case_id": "smoke_core_001",
    "features": {
        "intrinsic_safety": 0.58,
        "evidence_strength": 0.55,
        "bias_harm_index": 0.44,
        "uncertainty_calibration": 0.52,
        "traceability_integrity": 0.53,
    },
}

EXPLORATION_CASES = {
    "Archival contract (replay)": ARCHIVAL_CASE,
    "Low intrinsic safety (replay)": {
        "case_id": "explore_low_safety",
        "features": {
            "intrinsic_safety": 0.35,
            "evidence_strength": 0.60,
            "bias_harm_index": 0.40,
            "uncertainty_calibration": 0.58,
            "traceability_integrity": 0.57,
        },
    },
}


# 5. Core computation (modular) — engine only

All evaluation goes through **`evaluate_case`**; no gate formulas are reimplemented here.


In [3]:
def evaluate_replay(case: dict) -> dict:
    # Moderate profile, replay_mode via public engine API.
    return eng.evaluate_case(case, profile_name="moderate", mode=eng.MODE_REPLAY)


# 6. Interactive exploration (bounded, non-destructive)

Select a predefined scenario below. This changes on-screen results only; canonical outputs remain fixed.


In [4]:
explore_out = widgets.Output(layout={"border": "1px solid #ccc", "padding": "6px"})
dd = widgets.Dropdown(
    options=list(EXPLORATION_CASES.keys()),
    value="Archival contract (replay)",
    description="Scenario:",
    style={"description_width": "initial"},
)


def _render_exploration(label: str) -> None:
    with explore_out:
        clear_output(wait=True)
        case = EXPLORATION_CASES[label]
        r = evaluate_replay(case)
        print(f"Scenario: {label}  |  case_id={case['case_id']}")
        print(f"  governance_outcome: {r['governance_outcome']}")
        print(f"  approved: {r['approved']}  |  all_gates_pass: {r['all_gates_pass']}")
        print(f"  compensatory_score: {r['compensatory_score']}")


def _on_dd(change) -> None:
    if change.get("name") == "value" and change.get("new") is not None:
        _render_exploration(change["new"])


dd.observe(_on_dd, names="value")
_render_exploration(dd.value)
display(dd)
display(explore_out)


Dropdown(description='Scenario:', options=('Archival contract (replay)', 'Low intrinsic safety (replay)'), sty…

Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…

# 7. Output generation (strict contract)

The next cells write **only** `outputs/tables/smoke_test_results.csv` and `outputs/figures/smoke_test_summary.txt`, using **`ARCHIVAL_CASE` only** (not widget state). Serialisation matches the hash-locked manifest.


In [5]:
result = eng.evaluate_case(ARCHIVAL_CASE, profile_name="moderate", mode=eng.MODE_REPLAY)
result_hash = eng.hash_output(result)


In [6]:
csv_path = OUT_TAB / "smoke_test_results.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(
        [
            "case_id",
            "profile_name",
            "mode",
            "governance_outcome",
            "all_gates_pass",
            "compensatory_score",
            "compensatory_approved",
            "approved",
            "result_sha256",
        ]
    )
    w.writerow(
        [
            result["case_id"],
            result["profile_name"],
            result["mode"],
            result["governance_outcome"],
            result["all_gates_pass"],
            result["compensatory_score"],
            result["compensatory_approved"],
            result["approved"],
            result_hash,
        ]
    )


In [7]:
summary_path = OUT_FIG / "smoke_test_summary.txt"
lines = [
    "Shared-core smoke test (01_smoke_test)",
    "========================================",
    "",
    "Purpose: This notebook runs one fixed synthetic governance case through the",
    "corrected public engine v1.1 in replay_mode (five gates plus compensatory",
    "scoring only, matching the static historical replay path).",
    "",
    "Why this matters: A successful run proves the checked-out engine module",
    "executes end-to-end and that we can serialize a deterministic record under",
    "config/expected_outputs.json for automated manifest validation.",
    "",
    "Outputs produced (this notebook only):",
    "- outputs/tables/smoke_test_results.csv — tabular record for the case.",
    "- outputs/figures/smoke_test_summary.txt — this narrative summary.",
    "",
    f"Governance outcome:                  {result['governance_outcome']}",
    f"All gates pass (replay):             {result['all_gates_pass']}",
    f"Compensatory approved:               {result['compensatory_approved']}",
    f"Final approved flag (replay):       {result['approved']}",
    f"SHA-256 (canonical JSON result):    {result_hash}",
    "",
]
summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Smoke test artifacts written.")


Smoke test artifacts written.


# 8. Results interpretation

The **CSV** stores the archival verdict fields plus `result_sha256` from `hash_output`—the same digest the harness recomputes. The **summary file** mirrors those values for human readers. If any gate failed on `ARCHIVAL_CASE`, `governance_outcome` would read `REJECT`; here the frozen scenario is chosen so the replay path **approves**, exercising a successful smoke.


# 9. Reproducibility statement

- **Deterministic guarantee:** contract outputs depend only on `ARCHIVAL_CASE`, engine code, and moderate thresholds—not on widget state.
- **Restart & run all:** required for CI and `reproduce_all.py`; interactive cells run unattended with default dropdown selection.
- **Declared environment:** install from `requirements.txt` or conda env file; `ipywidgets` is declared for the exploration UI.
- **`reproduce_all.py`:** executes this notebook in plan order and validates digests in `config/expected_outputs.json`.

This notebook follows *Ten Simple Rules*-style structure: story, documented reasoning, modular engine calls, explicit inputs, bounded interactivity, strict artefact contract.
